# Masterclass: Advanced Data Visualization with Matplotlib & Seaborn
## Moving Beyond Basics to Professional-Grade Visuals
---
**Lecture Duration:** 3 Hours

### Objectives:
1.  **Understand Aesthetic Architecture:** How to control the global 'feel' of your plots.
2.  **High-Dimensional Storytelling:** Using Grids to compare multiple variables.
3.  **Statistical Precision:** Going beyond means to show distributions and individual points.
4.  **Complex Matrix Visualization:** Customizing heatmaps for clarity.
5.  **Layout Mastery:** Using `subplot_mosaic` for custom dashboard layouts.

## 1. Setup and Global Styling (30 mins)
A professional visualization starts with consistency. We use `rcParams` and `set_theme` to ensure every plot in our report looks like it belongs to the same study.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Set the 'whitegrid' style and a professional color palette
sns.set_theme(style="whitegrid", palette="muted", context="talk")

# Optional: High-resolution plots for Jupyter
%config InlineBackend.figure_format = 'retina'

print("Global styles applied!")

## 2. The Ridge Plot: Visualizing Distribution Shifts (45 mins)
When comparing a numeric variable across many categories (like 5+ types of diamond cuts), standard histograms overlap and become unreadable. Ridge plots solve this using a vertical stack of density plots.

In [ ]:
# Load the dataset
diamonds = sns.load_dataset("diamonds")

# We create a FacetGrid - an object that manages multiple subplots based on a variable
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
g = sns.FacetGrid(diamonds, row="cut", hue="cut", aspect=12, height=1.2, palette="viridis")

# Add KDE plots (the 'ridge')
g.map(sns.kdeplot, "price", bw_adjust=.5, clip_on=False, fill=True, alpha=1, linewidth=1.5)
g.map(sns.kdeplot, "price", clip_on=False, color="w", lw=2, bw_adjust=.5)

# Add a horizontal line for each 'ridge'
g.map(plt.axhline, y=0, lw=2, clip_on=False)

# Custom function to label the rows manually for a cleaner look
def label(x, color, label):
    ax = plt.gca()
    ax.text(0, .2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

g.map(label, "price")

# Remove overlap styling and axis labels for 'aesthetic' finish
g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)
plt.suptitle("Diamond Price Density by Cut Type", fontsize=18, y=0.98)
plt.show()

## 3. Data Integrity: Violin + Swarm Layering (45 mins)
Summary statistics (like bars) can be misleading. Violin plots show the distribution, but adding a 'Swarm' on top shows every single data point, providing ultimate transparency.

In [ ]:
penguins = sns.load_dataset("penguins")

plt.figure(figsize=(12, 8))

# 1. Create the Violin plot (The 'Shell')
# We set inner=None to remove the default boxplot inside
sns.violinplot(data=penguins, x="species", y="body_mass_g", color=".9", inner=None)

# 2. Layer the Swarm plot (The 'Details')
# 'dodge=True' separates the points by hue if applicable
sns.swarmplot(data=penguins, x="species", y="body_mass_g", hue="sex", palette="magma", size=4)

plt.title("Penguin Body Mass: Visualizing the Full Population", fontsize=16)
plt.legend(title="Sex", loc="upper left")
plt.show()

## 4. Relationship Discovery: PairGrids & JointPlots (30 mins)
How do all variables interact? We use JointPlots for individual pairs and PairGrids for a global view of correlations.

In [ ]:
# Hexagonal JointPlot for dense data
sns.jointplot(data=penguins, x="bill_length_mm", y="bill_depth_mm", 
              kind="hex", color="#4CB391", height=8)
plt.suptitle("Density of Bill Dimensions", y=1.02)
plt.show()

# Complex PairGrid: Upper and Lower triangles showing different views
g = sns.PairGrid(penguins, hue="species", corner=True) # corner=True removes the redundant top half
g.map_lower(sns.kdeplot, levels=4, color=".2")
g.map_lower(sns.scatterplot, s=10, alpha=0.6)
g.map_diag(sns.histplot, kde=True)
g.add_legend()
plt.show()

## 5. Heatmaps with Context (30 mins)
Raw heatmaps are often cluttered. We will apply a mask to the upper triangle (since correlation matrices are symmetrical) and use a diverging colormap.

In [ ]:
corr = penguins.corr(numeric_only=True)

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

f, ax = plt.subplots(figsize=(10, 8))

# Custom Diverging Palette (Low - Neutral - High)
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(corr, mask=mask, cmap=cmap, center=0, annot=True,
            square=True, linewidths=.5, cbar_kws={"shrink": .8})

plt.title("Correlation Matrix of Penguin Features", fontsize=16)
plt.show()